#travel planner

In [23]:
from dotenv import load_dotenv
load_dotenv()


False

In [24]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient(api_key='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')

@tool
def web_search(query: str) -> Dict[str, Any]:
    """search the web for the query and return the results"""
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

@tool
def place_search(query: str) -> Dict[str, Any]:
    """search for places based on the query and return the results"""
    try:
        return tavily_client.search(f"tourist attractions for sightseeing {query}")
    except Exception as e:
        return {"error": str(e)}

@tool
def hotel_search(query: str) -> Dict[str, Any]:
    """search for hotels based on the query and return the results"""
    try:
        return tavily_client.search(f"hotels for {query}")
    except Exception as e:
        return {"error": str(e)}

@tool
def suggest_destination(query: str) -> Dict[str, Any]:
    """suggest a destination based on the query and return the results"""
    try:
        return tavily_client.search(f"recommend destination tourist places {query}")
    except Exception as e:
        return {"error": str(e)}

@tool
def flight_search(query: str) -> Dict[str, Any]:
    """search for flights based on the query and return the results"""
    try:
        return tavily_client.search(f"flights for {query}")
    except Exception as e:
        return {"error": str(e)}

@tool
def train_search(query: str) -> Dict[str, Any]:
    """search for trains based on the query and return the results"""
    try:
        return tavily_client.search(f"trains for {query}")
    except Exception as e:
        return {"error": str(e)}

@tool
def bus_search(query: str) -> Dict[str, Any]:
    """search for buses based on the query and return the results"""
    try:
        return tavily_client.search(f"buses for {query}")
    except Exception as e:
        return {"error": str(e)}

@tool
def route_search(query: str) -> Dict[str, Any]:
    """search for routes based on the query and return the results"""
    try:
        return tavily_client.search(f"map routes for {query}")
    except Exception as e:
        return {"error": str(e)}


In [25]:
from langchain.agents import AgentState

class TravelState(AgentState):
    origin: str
    destination: str
    days: str
    budget: str
    preferred_mode: str
    preferred_stay: str
    place_type: str


In [31]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

MODEL = ChatOllama(model="llama3.1:8b", base_url="http://localhost:11434")


destination_agent = create_agent(
    model=MODEL,
    tools=[suggest_destination, web_search],
    system_prompt="""
    you are a destination specialist. the user has not picked a destination, only a
    place_type preference (e.g. hills, beach, solo-friendly). based on the origin and
    place_type, recommend the single best destination and briefly say why.
    do not ask follow up questions.
    """
)

transport_agent = create_agent(
    model=MODEL,
    tools=[flight_search, train_search, bus_search, route_search, web_search],
    system_prompt="""
    you are a transport specialist. based on the origin and destination, and the user's
    preferred_mode if given, find the best transport option by price and duration.
    if no preferred_mode is given, compare flight/train/bus and recommend the best overall.
    do not ask follow up questions. report a short shortlist clearly.
    """
)

hotel_agent = create_agent(
    model=MODEL,
    tools=[hotel_search],
    system_prompt="""
    you are a hotel specialist. based on the destination, budget, and preferred_stay,
    recommend 2-3 suitable hotels with approximate price and rating if available.
    do not ask follow up questions.
    """
)

sightseeing_agent = create_agent(
    model=MODEL,
    tools=[place_search],
    system_prompt="""
    you are a sightseeing specialist. based on the destination, recommend the best sites
    and scenic spots to visit. do not ask follow up questions. keep the list short and
    prioritized.
    """
)

restaurant_agent = create_agent(
    model=MODEL,
    tools=[web_search],
    system_prompt="""
    you are a restaurant specialist. based on the destination, recommend suitable
    restaurants and famous local food, with rough price range if available.
    do not ask follow up questions.
    """
)

musttry_agent = create_agent(
    model=MODEL,
    tools=[web_search],
    system_prompt="""
    you are a local-experience specialist. based on the destination, recommend must-try
    activities, temples/landmarks, and shopping spots. do not ask follow up questions.
    """
)

budget_agent = create_agent(
    model=MODEL,
    tools=[],
    system_prompt="""
    you are a budget specialist. you will be given transport, hotel, restaurant,
    sightseeing and must-try findings for a trip, plus days and the user's stated budget.
    extract every price mentioned, estimate missing categories sensibly, and produce a
    short expense breakdown (transport, stay, food, activities, total). flag clearly if
    the total is likely to exceed the stated budget.
    """
)


In [32]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def recommend_destination(runtime: ToolRuntime) -> str:
    """the recommend_destination tool suggests a destination from the origin based on the preference type specified by the user"""
    origin = runtime.state["origin"]
    place_type = runtime.state["place_type"]
    query = f"recommend a {place_type} destination from {origin}"
    response = destination_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def search_transport(runtime: ToolRuntime) -> str:
    """the search_transport tool analyses all the possible modes of transport"""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    mode = runtime.state["preferred_mode"]
    query = f"search for transport options from {origin} to {destination}, preferred mode: {mode}"
    response = transport_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def search_hotel(runtime: ToolRuntime) -> str:
    """the search_hotel tool searches for available hotels at the destination"""
    destination = runtime.state["destination"]
    stay = runtime.state["preferred_stay"]
    budget = runtime.state["budget"]
    query = f"search for {stay} hotels at {destination} within budget {budget}"
    response = hotel_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def search_sightseeing(runtime: ToolRuntime) -> str:
    """the search_sightseeing tool searches for sightseeing options at the destination"""
    destination = runtime.state["destination"]
    query = f"search for sightseeing options at {destination}"
    response = sightseeing_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def search_restaurant(runtime: ToolRuntime) -> str:
    """the search_restaurant tool searches for restaurants at the destination"""
    destination = runtime.state["destination"]
    query = f"search for restaurants at {destination}"
    response = restaurant_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def search_musttry(runtime: ToolRuntime) -> str:
    """the search_musttry tool searches for must-try experiences at the destination"""
    destination = runtime.state["destination"]
    query = f"search for must-try experiences at {destination}"
    response = musttry_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def calculate_budget(
    transport_findings: str,
    hotel_findings: str,
    restaurant_findings: str,
    sightseeing_findings: str,
    musttry_findings: str,
    runtime: ToolRuntime,
) -> str:
    """the calculate_budget tool calculates the total budget required for the trip based on the findings from the other tools"""
    days = runtime.state["days"]
    budget = runtime.state["budget"]
    query = f"""
    days: {days}
    stated budget: {budget}
    transport: {transport_findings}
    hotel: {hotel_findings}
    restaurant: {restaurant_findings}
    sightseeing: {sightseeing_findings}
    must-try: {musttry_findings}
    """
    response = budget_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response["messages"][-1].content

@tool
def update_state(
    origin: str = "",
    destination: str = "",
    days: str = "",
    budget: str = "",
    preferred_mode: str = "",
    preferred_stay: str = "",
    place_type: str = "",
    runtime: ToolRuntime = None,
) -> str:
    """update the state with any values you currently know. you may call this multiple
    times as you learn more - fields you don't pass keep their previous value."""
    current = runtime.state
    return Command(update={
        "origin": origin or current.get("origin", ""),
        "destination": destination or current.get("destination", ""),
        "days": days or current.get("days", ""),
        "budget": budget or current.get("budget", ""),
        "preferred_mode": preferred_mode or current.get("preferred_mode", ""),
        "preferred_stay": preferred_stay or current.get("preferred_stay", ""),
        "place_type": place_type or current.get("place_type", ""),
        "messages": [ToolMessage("successfully updated state", tool_call_id=runtime.tool_call_id)],
    })


In [33]:
from langchain.agents import create_agent

coordinator = create_agent(
    model=MODEL,
    tools=[
        update_state,
        recommend_destination,
        search_transport,
        search_hotel,
        search_sightseeing,
        search_restaurant,
        search_musttry,
        calculate_budget,
    ],
    state_schema=TravelState,
    system_prompt="""
    you are a travel coordinator. you have exactly these tools - use these exact names,
    never invent new ones:

    - update_state
    - recommend_destination
    - search_transport
    - search_hotel
    - search_sightseeing
    - search_restaurant
    - search_musttry
    - calculate_budget

    critical rule: you must never write your own answer to what a tool would return.
    if you need transport info, hotel info, restaurant info, etc., you must call the
    matching tool and wait for its result. writing the answer yourself without calling
    the tool is forbidden, even if you're confident.

    on every turn, call exactly one tool. do not describe multiple future steps in the
    same turn. before acting, check what is already known in the state.

    do this in order:
    1. call update_state with whatever values you can extract from the user's message.
       you do not need to include every field - omit what you don't know yet.
    2. only if destination is still empty: call recommend_destination, then call
       update_state again passing only destination with the chosen value. if destination
       is already known, skip this step entirely.
    3. call search_transport, search_hotel, search_sightseeing, search_restaurant, and
       search_musttry - one real tool call each.
    4. call calculate_budget with the findings from step 3.
    5. once all tools above are done, reply with your final answer as plain text using
       exactly this format. fill in every field with what you found. write
       "not specified" only if the user never gave it and no tool could reasonably
       provide it:

    Origin: <value>
    Destination: <value>
    Mode of Transport: <value>
    Estimated Budget: <value>

    Hotel Recommendations:
    - <hotel name> - <price/night> - <rating if available>
    - <hotel name> - <price/night> - <rating if available>

    Places to Visit:
    - <place 1>
    - <place 2>
    - <place 3>

    Restaurants:
    - <restaurant name> - <known for / price range>
    - <restaurant name> - <known for / price range>

    Must Try:
    - <activity / food / experience>
    - <activity / food / experience>

    Budget Breakdown:
    Transport: <amount>
    Stay: <amount>
    Food: <amount>
    Activities: <amount>
    Total: <amount>

    Notes:
    - <anything the user should keep in mind>
    """
)


In [34]:
from langchain.messages import HumanMessage

response = coordinator.invoke(
    {
        "messages": [HumanMessage(content="i'm from coimbatore, i'm planning for a 2 day trip to munnar, it is a solo trip within a budget of 3000 INR")],
    },
    config={"tags": ["travel-planner"], "recursion_limit": 40},
)


In [ ]:
from pprint import pprint

pprint(response)


In [ ]:
print(response["messages"][-1].content)
